In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import r2_score, mean_squared_error
import xgboost as xgb
import lightgbm as lgb
import optuna

/Users/richard/.pyenv/versions/tf310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def add_features(data):
    data = data.copy()
    data['Sample Date'] = pd.to_datetime(data['Sample Date'], format='%d-%m-%Y', errors='coerce')
    data = data.dropna(subset=['Sample Date'])
    
    data['lat'] = data['Latitude']
    data['lon'] = data['Longitude']
    
    # Trigonometric encoding for lat/lon
    for angle, prefix in [(data['lat'], 'lat'), (data['lon'], 'lon')]:
        rad = np.radians(angle)
        for k in [1, 2, 3, 4, 5]:
            data[f'{prefix}_sin_{k}'] = np.sin(k * rad)
            data[f'{prefix}_cos_{k}'] = np.cos(k * rad)
    
    # Time features
    data['year'] = data['Sample Date'].dt.year.astype(float)
    data['year_norm'] = (data['year'] - data['year'].min()) / (data['year'].max() - data['year'].min() + 1e-8)
    data['month'] = data['Sample Date'].dt.month.astype(float)
    data['doy']   = data['Sample Date'].dt.dayofyear.astype(float)
    
    # Cyclic encodings
    data['doy_sin_365'] = np.sin(2 * np.pi * data['doy'] / 365.25)
    data['doy_cos_365'] = np.cos(2 * np.pi * data['doy'] / 365.25)
    data['month_sin_12'] = np.sin(2 * np.pi * data['month'] / 12)
    data['month_cos_12'] = np.cos(2 * np.pi * data['month'] / 12)
    
    for period, label in [(182.625, '182_6'), (91.3125, '91_3'), (30.4375, '30_4'), (7.3, 'weekly')]:
        data[f'doy_sin_{label}'] = np.sin(2 * np.pi * data['doy'] / period)
        data[f'doy_cos_{label}'] = np.cos(2 * np.pi * data['doy'] / period)
    
    # Interaction features
    data['lat_doy_sin'] = data['lat_sin_1'] * data['doy_sin_365']
    data['lon_doy_sin'] = data['lon_sin_1'] * data['doy_sin_365']
    data['lat_year']    = data['lat'] * data['year_norm']
    data['lon_year']    = data['lon'] * data['year_norm']
    data['pet_lat']     = data.get('pet', 0) * data['lat']
    data['pet_doy_sin'] = data.get('pet', 0) * data['doy_sin_365']
    
    # Landsat band ratios
    epsilon = 1e-8
    data['turbidity_index'] = data.get('nir', 0) / (data.get('green', 0) + epsilon)
    data['phosphorus_index'] = (data.get('swir16', 0) - data.get('nir', 0)) / (data.get('swir16', 0) + data.get('nir', 0) + epsilon)
    data['ec_index'] = data.get('swir22', 0) / (data.get('swir16', 0) + epsilon)
    data['alkalinity_index'] = data.get('green', 0) / (data.get('swir22', 0) + epsilon)
    data['green_nir'] = data.get('green', 0) / (data.get('nir', 0) + epsilon)
    data['swir16_green'] = data.get('swir16', 0) / (data.get('green', 0) + epsilon)
    data['pet_ndmi'] = data.get('pet', 0) * data.get('NDMI', 0)
    data['pet_nir'] = data.get('pet', 0) * data.get('nir', 0)
    data['ndmi_swir22'] = data.get('NDMI', 0) * data.get('swir22', 0)
    
    return data

In [3]:
df_target = pd.read_csv('water_quality_training_dataset.csv')
df_terra = pd.read_csv('terraclimate_features_training.csv')
df_landsat = pd.read_csv('landsat_features_training.csv')
df_deltares = pd.read_csv('deltares_water_2010_2015_200_samples.csv')

# Load large extra Landsat datasets
df_landsat_extra1 = pd.read_csv('landsat_features_2000_2006_8000_samples.csv')
df_landsat_extra2 = pd.read_csv('landsat_features_2015_2023_8000.csv')

# Merge train
df_train = df_target.merge(df_terra, on=['Latitude', 'Longitude', 'Sample Date'], how='left')
df_train = df_train.merge(df_landsat, on=['Latitude', 'Longitude', 'Sample Date'], how='left')
df_train = df_train.merge(df_deltares, on=['Latitude', 'Longitude', 'Sample Date'], how='left')
df_train['is_train'] = 1

# Load sub
sub = pd.read_csv('submission_template.csv')
df_terra_val = pd.read_csv('terraclimate_features_validation.csv')
df_landsat_val = pd.read_csv('landsat_features_validation.csv')

sub = sub.merge(df_terra_val, on=['Latitude', 'Longitude', 'Sample Date'], how='left')
sub = sub.merge(df_landsat_val, on=['Latitude', 'Longitude', 'Sample Date'], how='left')
sub['is_train'] = 0
sub['Total Alkalinity'] = np.nan
sub['Electrical Conductance'] = np.nan
sub['Dissolved Reactive Phosphorus'] = np.nan

# Prepare extras for concatenation to df_all (to enrich lags)
extras = [df_landsat_extra1, df_landsat_extra2, df_deltares]  # Include deltares full for extra data points
df_extras = pd.concat(extras, ignore_index=True)
df_extras = add_features(df_extras)

# Add missing columns to extras
missing_cols = set(df_train.columns) - set(df_extras.columns)
for col in missing_cols:
    df_extras[col] = np.nan if col in ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus'] else 0
df_extras['is_train'] = 0

# Combine train and sub
df_all = pd.concat([df_train, sub], ignore_index=True)
df_all = add_features(df_all)

# Concat extras to df_all
df_all = pd.concat([df_all, df_extras], ignore_index=True)

# Remove duplicates if any
df_all = df_all.drop_duplicates(subset=['Latitude', 'Longitude', 'Sample Date'])

# Add lags per location
df_all = df_all.sort_values(['Latitude', 'Longitude', 'Sample Date'])
lag_features = ['pet', 'nir', 'green', 'swir16', 'swir22', 'NDMI', 'MNDWI', 'turbidity_index', 'phosphorus_index', 'ec_index', 'P', 'ETa', 'PET', 'Temp']
group_key = ['Latitude', 'Longitude']

for lag in [1, 2, 3, 7]:
    for f in lag_features:
        shift_series = df_all.groupby(group_key)[f].shift(lag)
        df_all[f'{f}_lag{lag}'] = shift_series.values
        df_all[f'{f}_delta{lag}'] = (df_all[f] - shift_series).values
        df_all[f'{f}_ratio{lag}'] = (df_all[f] / (shift_series + 1e-8)).values

# Rolling means
for win in [3, 7]:
    for f in lag_features:
        roll_mean = df_all.groupby(group_key)[f].rolling(win, min_periods=1).mean().reset_index(0, drop=True)
        df_all[f'{f}_roll{win}'] = roll_mean.values

df_all = df_all.fillna(0)

# Update features
lag_feats = [c for c in df_all.columns if '_lag' in c or '_delta' in c or '_ratio' in c or '_roll' in c]
raw_features = ['pet', 'nir', 'green', 'swir16', 'swir22', 'NDMI', 'MNDWI',
                'P', 'ETa', 'PET', 'Melt', 'Snow', 'Temp', 'P_res', 'S_res', 
                'Ea_res', 'Qin_res', 'FracFull', 'Qout_res', 
                'turbidity_index', 'phosphorus_index', 'ec_index', 'alkalinity_index', 
                'green_nir', 'swir16_green', 'pet_ndmi', 'pet_nir', 'ndmi_swir22'] + lag_feats

engineered = [c for c in df_all.columns if any(x in c for x in [
    'sin', 'cos', 'year', 'doy_', 'month_', 'lat_', 'lon_', 'pet_lat', 'pet_doy_sin'
])]

features = raw_features + engineered

# Prepare train
df_train = df_all[df_all['is_train'] == 1].copy()
X = df_train[features]
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)

targets = {
    'TA':  np.log1p(df_train['Total Alkalinity']),
    'EC':  np.log1p(df_train['Electrical Conductance']),
    'DRP': np.log1p(df_train['Dissolved Reactive Phosphorus'])
}

/var/folders/hv/kxrxx3qn2rz5xqvmhb82grsw0000gn/T/ipykernel_29999/3137874531.py:44: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_all = pd.concat([df_all, df_extras], ignore_index=True)
/var/folders/hv/kxrxx3qn2rz5xqvmhb82grsw0000gn/T/ipykernel_29999/3137874531.py:58: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_all[f'{f}_delta{lag}'] = (df_all[f] - shift_series).values
/var/folders/hv/kxrxx3qn2rz5xqvmhb82grsw0000gn/T/ipykernel_29999/3137874531.py:59: PerformanceWarning: DataFrame is highly fragmented.  Th

In [4]:
def objective(trial, X_scaled, y, model_type):
    if model_type == 'xgb':
        params = {
            'objective': 'reg:squarederror',
            'eval_metric': 'rmse',
            'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.02),
            'max_depth': trial.suggest_int('max_depth', 4, 9),
            'subsample': trial.suggest_float('subsample', 0.6, 0.95),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.95),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 15),
            'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 2.0),
            'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 2.0),
            'tree_method': 'hist',
            'seed': 42
        }
        num_boost_round = 5000
    else:  # lgb
        params = {
            'objective': 'regression',
            'metric': 'rmse',
            'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.02),
            'max_depth': trial.suggest_int('max_depth', 4, 9),
            'num_leaves': trial.suggest_int('num_leaves', 15, 255),
            'subsample': trial.suggest_float('subsample', 0.6, 0.95),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.95),
            'min_child_weight': trial.suggest_float('min_child_weight', 0.001, 0.2),
            'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 2.0),
            'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 2.0),
            'n_jobs': -1,
            'verbosity': -1,
            'random_state': 42,
            'force_col_wise': True
        }
    
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    oof = np.zeros(len(y))
    
    for tr_idx, val_idx in kf.split(X_scaled):
        X_tr, X_val = X_scaled[tr_idx], X_scaled[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        
        if model_type == 'xgb':
            dtrain = xgb.DMatrix(X_tr, label=y_tr)
            dval = xgb.DMatrix(X_val, label=y_val)
            booster = xgb.train(
                params,
                dtrain,
                num_boost_round=5000,
                evals=[(dval, 'eval')],
                early_stopping_rounds=200,
                verbose_eval=False
            )
            pred = booster.predict(dval)
        else:
            model = lgb.LGBMRegressor(**params)
            model.fit(
                X_tr, y_tr,
                eval_set=[(X_val, y_val)],
                callbacks=[lgb.early_stopping(200, verbose=False)]
            )
            pred = model.predict(X_val)
        
        oof[val_idx] = pred
    
    return np.sqrt(mean_squared_error(y, oof))

# Tune for each target and model
best_params = {}
for name, y in targets.items():
    print(f"\nTuning for {name}")
    
    # Tune XGBoost
    study_xgb = optuna.create_study(direction='minimize')
    study_xgb.optimize(lambda trial: objective(trial, X_scaled, y, 'xgb'), n_trials=50)
    best_params[f'{name}_xgb'] = study_xgb.best_params
    
    # Tune LightGBM
    study_lgb = optuna.create_study(direction='minimize')
    study_lgb.optimize(lambda trial: objective(trial, X_scaled, y, 'lgb'), n_trials=50)
    best_params[f'{name}_lgb'] = study_lgb.best_params


[I 2026-03-04 17:58:56,844] A new study created in memory with name: no-name-da4a2875-022d-4168-a0c7-5519ec4af071



Tuning for TA


[I 2026-03-04 17:59:46,652] Trial 0 finished with value: 0.30993462989929715 and parameters: {'learning_rate': 0.017905031299389862, 'max_depth': 5, 'subsample': 0.7825796113319611, 'colsample_bytree': 0.8737192391477695, 'min_child_weight': 8, 'reg_alpha': 0.11457259653152785, 'reg_lambda': 1.317101472113556}. Best is trial 0 with value: 0.30993462989929715.
[I 2026-03-04 18:00:55,500] Trial 1 finished with value: 0.3279271693024588 and parameters: {'learning_rate': 0.0016474490558340642, 'max_depth': 6, 'subsample': 0.8202760377885056, 'colsample_bytree': 0.7306219456400711, 'min_child_weight': 8, 'reg_alpha': 1.8824732975947065, 'reg_lambda': 1.2403655857416505}. Best is trial 0 with value: 0.30993462989929715.
[I 2026-03-04 18:01:54,140] Trial 2 finished with value: 0.30624527405567803 and parameters: {'learning_rate': 0.017410494048146985, 'max_depth': 7, 'subsample': 0.7347900827107127, 'colsample_bytree': 0.6351419227621484, 'min_child_weight': 12, 'reg_alpha': 0.064812556544472


Tuning for EC


[I 2026-03-04 19:16:48,124] Trial 0 finished with value: 0.2610505366913159 and parameters: {'learning_rate': 0.011484881664723624, 'max_depth': 8, 'subsample': 0.8183621689320874, 'colsample_bytree': 0.6666568406255361, 'min_child_weight': 10, 'reg_alpha': 0.7530254895564796, 'reg_lambda': 0.5352725541244847}. Best is trial 0 with value: 0.2610505366913159.
[I 2026-03-04 19:17:51,579] Trial 1 finished with value: 0.27030526565879603 and parameters: {'learning_rate': 0.01830379336246378, 'max_depth': 7, 'subsample': 0.7179167192289512, 'colsample_bytree': 0.7355756044458129, 'min_child_weight': 8, 'reg_alpha': 1.664887932035124, 'reg_lambda': 1.1262296204992996}. Best is trial 0 with value: 0.2610505366913159.
[I 2026-03-04 19:18:59,668] Trial 2 finished with value: 0.27028255267911744 and parameters: {'learning_rate': 0.005527261528298895, 'max_depth': 6, 'subsample': 0.8072634096867929, 'colsample_bytree': 0.6288011608581874, 'min_child_weight': 12, 'reg_alpha': 1.1296100596518543, '


Tuning for DRP


[I 2026-03-04 20:22:17,050] Trial 0 finished with value: 0.5367688928141555 and parameters: {'learning_rate': 0.0067927767594183315, 'max_depth': 9, 'subsample': 0.8958197977747031, 'colsample_bytree': 0.6829750603339294, 'min_child_weight': 14, 'reg_alpha': 1.4773708537349683, 'reg_lambda': 0.6497157875125417}. Best is trial 0 with value: 0.5367688928141555.
[I 2026-03-04 20:22:51,585] Trial 1 finished with value: 0.5370263533791332 and parameters: {'learning_rate': 0.01798961838541424, 'max_depth': 8, 'subsample': 0.8465504255536248, 'colsample_bytree': 0.8554536328470338, 'min_child_weight': 3, 'reg_alpha': 0.5063676824274261, 'reg_lambda': 1.8416299937358058}. Best is trial 0 with value: 0.5367688928141555.
[I 2026-03-04 20:24:11,497] Trial 2 finished with value: 0.5430292504893572 and parameters: {'learning_rate': 0.005608367314822063, 'max_depth': 7, 'subsample': 0.7353758193735042, 'colsample_bytree': 0.8045604325684718, 'min_child_weight': 14, 'reg_alpha': 0.8082972358549421, '

In [5]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
models = {}

for name, y in targets.items():
    print(f"\n=== Training {name} with best params ===")
    oof = np.zeros(len(y))
    xgb_list = []
    lgb_list = []
    
    xgb_params = best_params[f'{name}_xgb']
    xgb_params.update({'objective': 'reg:squarederror', 'eval_metric': 'rmse', 'tree_method': 'hist', 'seed': 42})
    
    lgb_params = best_params[f'{name}_lgb']
    lgb_params.update({'objective': 'regression', 'metric': 'rmse', 'n_jobs': -1, 'verbosity': -1, 'random_state': 42, 'force_col_wise': True})
    
    for fold, (tr_idx, val_idx) in enumerate(kf.split(X_scaled)):
        X_tr, X_val = X_scaled[tr_idx], X_scaled[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        
        # XGBoost
        dtrain = xgb.DMatrix(X_tr, label=y_tr)
        dval = xgb.DMatrix(X_val, label=y_val)
        booster = xgb.train(
            xgb_params,
            dtrain,
            num_boost_round=5000,
            evals=[(dval, 'eval')],
            early_stopping_rounds=200,
            verbose_eval=200
        )
        xgb_list.append(booster)
        
        # LightGBM
        lgb_model = lgb.LGBMRegressor(**lgb_params)
        lgb_model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(200)]
        )
        lgb_list.append(lgb_model)
        
        # OOF
        xgb_pred = booster.predict(dval)
        lgb_pred = lgb_model.predict(X_val)
        blend_pred = (xgb_pred * 0.5  + lgb_pred * 0.5)  # Equal blend
        oof[val_idx] = blend_pred
    
    orig_y = df_train[name.replace('TA', 'Total Alkalinity').replace('EC', 'Electrical Conductance').replace('DRP', 'Dissolved Reactive Phosphorus')]
    oof_back = np.expm1(oof)
    r2 = r2_score(orig_y, oof_back)
    rmse = np.sqrt(mean_squared_error(orig_y, oof_back))
    print(f"{name}  CV R² = {r2:.4f}   RMSE = {rmse:.2f}")
    
    models[name] = (xgb_list, lgb_list)


=== Training TA with best params ===
[0]	eval-rmse:0.85774
[200]	eval-rmse:0.36330
[400]	eval-rmse:0.29675
[600]	eval-rmse:0.28636
[800]	eval-rmse:0.28389
[1000]	eval-rmse:0.28274
[1200]	eval-rmse:0.28212
[1400]	eval-rmse:0.28181
[1600]	eval-rmse:0.28146
[1800]	eval-rmse:0.28130
[1963]	eval-rmse:0.28129
Training until validation scores don't improve for 200 rounds
Did not meet early stopping. Best iteration is:
[100]	valid_0's rmse: 0.346126
[0]	eval-rmse:0.87567


/Users/richard/.pyenv/versions/tf310/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[200]	eval-rmse:0.38309
[400]	eval-rmse:0.31720
[600]	eval-rmse:0.30687
[800]	eval-rmse:0.30453
[1000]	eval-rmse:0.30356
[1200]	eval-rmse:0.30298
[1400]	eval-rmse:0.30241
[1600]	eval-rmse:0.30213
[1800]	eval-rmse:0.30193
[2000]	eval-rmse:0.30178
[2200]	eval-rmse:0.30163
[2400]	eval-rmse:0.30149
[2600]	eval-rmse:0.30145
[2800]	eval-rmse:0.30141
[3000]	eval-rmse:0.30137
[3179]	eval-rmse:0.30136
Training until validation scores don't improve for 200 rounds
Did not meet early stopping. Best iteration is:
[100]	valid_0's rmse: 0.360073
[0]	eval-rmse:0.79949


/Users/richard/.pyenv/versions/tf310/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[200]	eval-rmse:0.35982
[400]	eval-rmse:0.30441
[600]	eval-rmse:0.29410
[800]	eval-rmse:0.29107
[1000]	eval-rmse:0.28992
[1200]	eval-rmse:0.28891
[1400]	eval-rmse:0.28823
[1600]	eval-rmse:0.28764
[1800]	eval-rmse:0.28727
[2000]	eval-rmse:0.28691
[2200]	eval-rmse:0.28665
[2400]	eval-rmse:0.28646
[2600]	eval-rmse:0.28634
[2800]	eval-rmse:0.28627
[3000]	eval-rmse:0.28619
[3200]	eval-rmse:0.28613
[3400]	eval-rmse:0.28607
[3600]	eval-rmse:0.28605
[3800]	eval-rmse:0.28603
[3917]	eval-rmse:0.28603
Training until validation scores don't improve for 200 rounds
Did not meet early stopping. Best iteration is:
[100]	valid_0's rmse: 0.342889
[0]	eval-rmse:0.84336


...

/Users/richard/.pyenv/versions/tf310/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[200]	eval-rmse:0.61352
[400]	eval-rmse:0.53791
[600]	eval-rmse:0.51724
[800]	eval-rmse:0.51043
[1000]	eval-rmse:0.50853
[1200]	eval-rmse:0.50765
[1400]	eval-rmse:0.50693
[1600]	eval-rmse:0.50664
[1800]	eval-rmse:0.50637
[2000]	eval-rmse:0.50618
[2200]	eval-rmse:0.50614
[2400]	eval-rmse:0.50607
[2600]	eval-rmse:0.50602
[2800]	eval-rmse:0.50605
[2842]	eval-rmse:0.50603
Training until validation scores don't improve for 200 rounds
Did not meet early stopping. Best iteration is:
[100]	valid_0's rmse: 0.56011
[0]	eval-rmse:0.92574


/Users/richard/.pyenv/versions/tf310/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[200]	eval-rmse:0.64182
[400]	eval-rmse:0.57636
[600]	eval-rmse:0.55868
[800]	eval-rmse:0.55231
[1000]	eval-rmse:0.54981
[1200]	eval-rmse:0.54855
[1400]	eval-rmse:0.54785
[1600]	eval-rmse:0.54688
[1800]	eval-rmse:0.54641
[2000]	eval-rmse:0.54597
[2200]	eval-rmse:0.54574
[2400]	eval-rmse:0.54545
[2600]	eval-rmse:0.54546
[2643]	eval-rmse:0.54545
Training until validation scores don't improve for 200 rounds
Did not meet early stopping. Best iteration is:
[100]	valid_0's rmse: 0.596011
DRP  CV R² = 0.5929   RMSE = 32.53


/Users/richard/.pyenv/versions/tf310/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [6]:
sub = df_all[df_all['is_train'] == 0].copy()

sub_X = sub[features]
sub_X_scaled = scaler.transform(sub_X)
dsub = xgb.DMatrix(sub_X_scaled)

def blend_predict(xgb_models, lgb_models, X, dX):
    xgb_preds = np.mean([m.predict(dX) for m in xgb_models], axis=0)
    lgb_preds = np.mean([m.predict(X) for m in lgb_models], axis=0)
    return (xgb_preds * 0.5 + lgb_preds * 0.5)

sub['Total Alkalinity'] = np.expm1(blend_predict(models['TA'][0], models['TA'][1], sub_X_scaled, dsub))
sub['Electrical Conductance'] = np.expm1(blend_predict(models['EC'][0], models['EC'][1], sub_X_scaled, dsub))
sub['Dissolved Reactive Phosphorus'] = np.expm1(blend_predict(models['DRP'][0], models['DRP'][1], sub_X_scaled, dsub))

# Clip
sub['Total Alkalinity'] = sub['Total Alkalinity'].clip(5, 400)
sub['Electrical Conductance'] = sub['Electrical Conductance'].clip(30, 2200)
sub['Dissolved Reactive Phosphorus'] = sub['Dissolved Reactive Phosphorus'].clip(0.5, 250)

final_sub = sub[['Latitude', 'Longitude', 'Sample Date',
                 'Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']]

final_sub.to_csv('submission_large_extras_lag_blend.csv', index=False)
print("✅ Saved → submission_large_extras_lag_blend.csv")
print("Upload this file and check the leaderboard!")

✅ Saved → submission_large_extras_lag_blend.csv
Upload this file and check the leaderboard!


/Users/richard/.pyenv/versions/tf310/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/richard/.pyenv/versions/tf310/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/richard/.pyenv/versions/tf310/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/richard/.pyenv/versions/tf310/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/richard/.pyenv/versions/tf310/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid featu